# Vision Assist V2 — Voice Demo (notebook runner)

Runs the reusable `vision_assist` package on 8 ROD validation images with the **existing**
YOLOv8n checkpoint (`models/rod_yolov8n_50ep_best.pt`). No training happens here.
The images are listed in `configs/demo_images.txt`: six scenes with several hazards, plus one
stairs and one manhole image, chosen from ground-truth labels by `scripts/select_demo_images.py`.

Outputs (under `outputs/demo/` inside the repo): annotated frames, `vision_assist_demo.wav`,
`vision_assist_demo.mp4`, `transcript.txt` and `demo_results.json` (detections, alerts, latency).

**Kaggle:** attach the *Obstacle Detection Dataset* (ROD, `abtinzandi/obstacle-detection-dataset`)
via *Add Input*, enable Internet, then run all cells. GPU is optional.
**Elsewhere:** run from the repo root and set `IMAGE_SOURCE` to any image folder.

This is an image slideshow, not a live camera system, and not a safety-validated navigation aid.

## 1. Get the code
On Kaggle the repository is cloned into `/kaggle/working`. Locally, the notebook uses the checkout it lives in.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/SudhaMsr/VisionAssist-MVP.git"
BRANCH = "main"  # change to a feature branch to test unmerged work

if Path("/kaggle/working").exists():
    REPO_DIR = Path("/kaggle/working/VisionAssist-MVP")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

assert (REPO_DIR / "vision_assist").is_dir(), f"vision_assist package not found under {REPO_DIR}"
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)

## 2. Dependencies
Python packages from `requirements.txt`; eSpeak NG (offline TTS) and ffmpeg from apt if missing.

In [ ]:
import shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

missing = [pkg for pkg, exe in (("espeak-ng", "espeak-ng"), ("ffmpeg", "ffmpeg")) if shutil.which(exe) is None]
if missing:
    proc = subprocess.run(
        ["bash", "-c", f"apt-get update -qq && apt-get install -y -qq {' '.join(missing)}"],
        capture_output=True, text=True,
    )
    if proc.returncode != 0:
        print(proc.stderr[-1500:])
print("espeak-ng:", shutil.which("espeak-ng"))
print("ffmpeg:", shutil.which("ffmpeg"))

## 3. Run the test suite
Fast (well under a minute on CPU). Stop here if anything fails.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-3000:], result.stderr[-1500:])
assert result.returncode == 0, "Tests failed; see output above."

## 4. Run the demo

In [ ]:
from vision_assist.config import (
    DEFAULT_IMAGE_LIST, list_images, read_image_list, resolve_image_dir, resolve_model_path, resolve_output_dir,
)
from vision_assist.demo import run_demo

IMAGE_SOURCE = None   # None = $VISION_ASSIST_IMAGES or the Kaggle ROD valid/images folder
IMAGE_LIST = DEFAULT_IMAGE_LIST  # curated ROD validation images; None = first IMAGE_LIMIT in sorted order
IMAGE_LIMIT = 8

image_dir = resolve_image_dir(IMAGE_SOURCE)
images = read_image_list(IMAGE_LIST, image_dir) if IMAGE_LIST else list_images(image_dir, limit=IMAGE_LIMIT)
OUT_DIR = resolve_output_dir(None)
summary = run_demo(images, resolve_model_path(None), OUT_DIR, conf=0.35, max_alerts=2, cooldown_s=6.0)

## 5. Inspect outputs

In [ ]:
import json
from IPython.display import Audio, Image, Video, display

print(json.dumps(summary["latency"], indent=2))
print(json.dumps(summary["outputs"], indent=2))
print((OUT_DIR / "transcript.txt").read_text())
display(Image(filename=str(OUT_DIR / "frames" / "frame_01.jpg"), width=720))
display(Audio(filename=str(OUT_DIR / "vision_assist_demo.wav")))
if (OUT_DIR / "vision_assist_demo.mp4").exists():
    display(Video(str(OUT_DIR / "vision_assist_demo.mp4"), embed=True, width=720))

## Scope and limitations
- Uses the existing checkpoint; no retraining.
- Left/centre/right = which third of the **image** the box centre is in. It is not a direction for the user.
- Box size only orders alerts within a priority tier. It is **not** a distance estimate.
- No alert does **not** mean the path is clear.
- Not a live camera pipeline; not safety-validated.

Generated media in `outputs/` is git-ignored. Attach it to a release or PR rather than committing it.